# Day 9 — Reproducibility and Project Integrity Audit

**Audit only.** Modeling and evaluation are complete through Day 8. This notebook inspects existing files, Git history, and already-reported numbers. It does **not** train, tune, predict, or change the locked Random Forest.

In [ ]:
from pathlib import Path
import json
import subprocess

import pandas as pd

here = Path.cwd().resolve()
if (here / "notebooks" / "05_random_forest.ipynb").exists():
    ROOT = here
elif (here / "05_random_forest.ipynb").exists():
    ROOT = here.parent
else:
    ROOT = Path("..").resolve()
    if not (ROOT / "notebooks").exists():
        ROOT = Path(".").resolve()
NB_DIR = ROOT / "notebooks"

print("Audit root:", ROOT)
print("This session does not fit classifiers, reconstruct models, or generate predictions.")


## Section 1 — Project notebook inventory

In [ ]:
expected_globs = [
    "01_*.ipynb",
    "02_*.ipynb",
    "03_*.ipynb",
    "04_*.ipynb",
    "05_random_forest.ipynb",
    "06_model_analysis.ipynb",
    "07_*.ipynb",
    "08_final_evaluation.ipynb",
    "09_reproducibility_audit.ipynb",
]

rows = []
matched = set()
for pattern in expected_globs:
    hits = sorted(NB_DIR.glob(pattern))
    if not hits:
        rows.append(
            {
                "Expected pattern": pattern,
                "Notebook path": "(none)",
                "Exists": False,
                "Size (bytes)": None,
            }
        )
        continue
    for p in hits:
        matched.add(p.name)
        rows.append(
            {
                "Expected pattern": pattern,
                "Notebook path": str(p.relative_to(ROOT)).replace("\\", "/"),
                "Exists": True,
                "Size (bytes)": p.stat().st_size,
            }
        )

day9_notebook_inventory = pd.DataFrame(rows)
print("Expected notebook inventory")
display(day9_notebook_inventory)

all_nbs = sorted(NB_DIR.glob("*.ipynb"))
unexpected = [p for p in all_nbs if p.name not in matched]
print()
if unexpected:
    print("Unexpected notebook files:")
    for p in unexpected:
        print(f"  {p.relative_to(ROOT)} ({p.stat().st_size} bytes)")
else:
    print("No unexpected .ipynb files beyond the expected patterns.")

print()
print("Note: notebooks/03_*.ipynb is absent. Days 2–3 work lives in notebooks/02_baseline_model.ipynb.")


## Section 2 — Git / repository state

Read-only Git inspection. **No commit. No push. No file modification** from this audit except creating this Day 9 notebook itself (untracked until the user commits).

In [ ]:
def git(*args):
    r = subprocess.run(
        ["git", *args],
        cwd=ROOT,
        capture_output=True,
        text=True,
        check=False,
    )
    return r.stdout.strip(), r.stderr.strip(), r.returncode

branch, _, _ = git("rev-parse", "--abbrev-ref", "HEAD")
status, _, _ = git("status", "-sb")
ahead_behind, _, _ = git("rev-list", "--left-right", "--count", "origin/main...HEAD")
log, _, _ = git("log", "-12", "--oneline")

print("Current branch:", branch)
print("Status:")
print(status)
print()
print("origin/main...HEAD left-right counts (behind\\tahead):", ahead_behind)
print()
print("Recent commits:")
print(log)
print()

log_text = log.lower()
print("Days 5–8 appear committed:")
print("  Day 5 (random forest):", "day 5" in log_text or "random forest" in log_text)
print("  Day 6 (model analysis):", "day 6" in log_text)
print("  Day 7 (final model comparison):", "day 7" in log_text)
print("  Day 8 (final evaluation):", "day 8" in log_text)
print()
print(
    "Commit-message convention: Day 6 and Day 8 match 'feat: add day N ...'. "
    "Day 7 is 'feat: add day 7 final model comparison'. "
    "Day 5 is 'Add Day 5 Random Forest experiments' (same content, not the exact feat: prefix)."
)
print("This cell did not create a commit or push.")


## Section 3 — Day 5 locked model audit

Evidence is taken from existing notebook **source and saved outputs**, not from fitting a model.

In [ ]:
def notebook_text(path):
    p = NB_DIR / path
    if not p.exists():
        return ""
    nb = json.loads(p.read_text(encoding="utf-8"))
    chunks = []
    for cell in nb.get("cells", []):
        chunks.append("".join(cell.get("source", [])))
        for out in cell.get("outputs", []) or []:
            if out.get("output_type") == "stream":
                t = out.get("text", [])
                chunks.append("".join(t) if isinstance(t, list) else str(t))
            data = out.get("data") or {}
            for key in ("text/plain", "text/html", "text/markdown"):
                if key in data:
                    v = data[key]
                    chunks.append("".join(v) if isinstance(v, list) else str(v))
    return "\n".join(chunks)


def present(haystack, needle):
    return needle in haystack


d5 = notebook_text("05_random_forest.ipynb")
lock_checks = [
    ("Random Forest named as locked model", present(d5, "Random Forest") and present(d5, "500 trees")),
    ("n_estimators = 500", present(d5, "n_estimators=500") or present(d5, "n_estimators: 500")),
    ("class_weight=None", present(d5, "class_weight=None") or present(d5, "class_weight: None")),
    ("No SMOTE on locked RF", present(d5, "No SMOTE") or present(d5, "no SMOTE")),
    ("No threshold tuning", present(d5, "No threshold tuning") or present(d5, "no threshold tuning")),
    ("Selected on KDDTrain+ validation only", present(d5, "validation only")),
    ("Validation Macro F1 0.951031", present(d5, "0.951031")),
    ("KDDTest+ not used for selection", present(d5, "KDDTest+ has not been used for model selection")),
]

day9_lock_audit = pd.DataFrame(
    [{"Item": k, "Supported by Day 5 notebook evidence": v} for k, v in lock_checks]
)
display(day9_lock_audit)
print()
print("No Random Forest was fitted or reconstructed in this audit.")


## Section 4 — Validation metric consistency

In [ ]:
val_needles = {
    "Accuracy 0.998809": "0.998809",
    "Macro Precision 0.974485": "0.974485",
    "Macro Recall 0.931597": "0.931597",
    "Macro F1 0.951031": "0.951031",
    "R2L F1 0.982097": "0.982097",
    "U2R F1 0.777778": "0.777778",
}
val_nbs = {
    "05_random_forest.ipynb": notebook_text("05_random_forest.ipynb"),
    "06_model_analysis.ipynb": notebook_text("06_model_analysis.ipynb"),
    "07_final_model_comparison.ipynb": notebook_text("07_final_model_comparison.ipynb"),
    "08_final_evaluation.ipynb": notebook_text("08_final_evaluation.ipynb"),
}
val_rows = []
for label, needle in val_needles.items():
    row = {"Metric": label}
    for name, text in val_nbs.items():
        row[name] = "reported" if needle in text else "not explicitly stored/reported"
    val_rows.append(row)

day9_validation_consistency = pd.DataFrame(val_rows)
display(day9_validation_consistency)
print("Values were not recomputed by fitting or predicting.")


## Section 5 — KDDTest+ metric consistency

In [ ]:
test_needles = {
    "Accuracy 0.7447": "0.7447",
    "Macro Precision 0.8198": "0.8198",
    "Macro Recall 0.4896": "0.4896",
    "Macro F1 0.5061": "0.5061",
    "Normal P/R/F1 0.642372 / 0.973844 / 0.774117": "0.642372",
    "DoS P/R/F1 0.961345 / 0.770107 / 0.855165": "0.961345",
    "Probe P/R/F1 0.850000 / 0.596861 / 0.701286": "0.596861",
    "R2L P/R/F1 0.978571 / 0.047487 / 0.090579": "0.978571",
    "U2R P/R/F1 0.666667 / 0.059701 / 0.109589": "0.059701",
}
test_rows = []
for label, needle in test_needles.items():
    row = {"Metric": label}
    for name, text in val_nbs.items():
        row[name] = "reported" if needle in text else "not explicitly stored/reported"
    test_rows.append(row)

day9_kddtest_consistency = pd.DataFrame(test_rows)
display(day9_kddtest_consistency)
print("Predictions were not regenerated.")
print(
    "Note: Day 7 reports overall KDDTest+ macros and rounded R2L/U2R scores "
    "(e.g. R2L recall 0.0475). Full six-decimal per-class values are in Days 5, 6, and 8."
)


## Section 6 — Confusion matrix integrity

Audit of the **already-reported** Day 5 matrix. Row sums are checked against reported supports. The matrix is **not** rebuilt from `y_test_pred_rf`.

In [ ]:
cm = pd.DataFrame(
    [
        [9457, 67, 186, 0, 1],
        [1649, 5745, 66, 0, 0],
        [812, 164, 1445, 0, 0],
        [2744, 0, 3, 137, 1],
        [60, 0, 0, 3, 4],
    ],
    index=["Actual Normal", "Actual DoS", "Actual Probe", "Actual R2L", "Actual U2R"],
    columns=["Pred Normal", "Pred DoS", "Pred Probe", "Pred R2L", "Pred U2R"],
)
supports = cm.sum(axis=1)
print("Reported matrix")
display(cm)
print()
print("Row totals (actual support):")
display(supports.to_frame("Support"))
print("Total:", int(cm.values.sum()))
print("R2L support:", int(supports.loc["Actual R2L"]), "(expected 2885)")
print("U2R support:", int(supports.loc["Actual U2R"]), "(expected 67)")
print("Total == 22544:", int(cm.values.sum()) == 22544)
print("Normal support == 9711:", int(supports.loc["Actual Normal"]) == 9711)
print("DoS support == 7460:", int(supports.loc["Actual DoS"]) == 7460)
print("Probe support == 2421:", int(supports.loc["Actual Probe"]) == 2421)

cm_needles = ["9457", "2744", "5745", "1445"]
print()
for name, text in val_nbs.items():
    ok = all(n in text for n in cm_needles)
    print(f"{name}: characteristic CM counts present = {ok}")


## Section 7 — Generalization-gap consistency

In [ ]:
macro_abs = 0.951031 - 0.5061
macro_rel = macro_abs / 0.951031 * 100
r2l_drop = 0.964824 - 0.047487
u2r_drop = 0.700000 - 0.059701

day9_gap_check = pd.DataFrame(
    [
        {
            "Quantity": "Macro F1 absolute drop",
            "Expected": 0.4449,
            "Recomputed from reported pair": round(macro_abs, 4),
            "Match": abs(macro_abs - 0.4449) < 5e-5,
        },
        {
            "Quantity": "Macro F1 relative drop (%)",
            "Expected": 46.78,
            "Recomputed from reported pair": round(macro_rel, 2),
            "Match": abs(macro_rel - 46.78) < 0.01,
        },
        {
            "Quantity": "R2L recall drop",
            "Expected": 0.9173,
            "Recomputed from reported pair": round(r2l_drop, 4),
            "Match": abs(r2l_drop - 0.9173) < 5e-5,
        },
        {
            "Quantity": "U2R recall drop",
            "Expected": 0.6403,
            "Recomputed from reported pair": round(u2r_drop, 4),
            "Match": abs(u2r_drop - 0.6403) < 5e-5,
        },
    ]
)
display(day9_gap_check)
print("Arithmetic uses the already-reported metric pair only. No new predictions.")


## Section 8 — Day 6 distribution-shift evidence

In [ ]:
d6 = val_nbs["06_model_analysis.ipynb"]
shift_items = [
    ("R2L validation prevalence 0.7898%", "0.7898"),
    ("R2L KDDTest+ prevalence 12.7972%", "12.7972"),
    ("U2R validation prevalence 0.0397%", "0.0397"),
    ("U2R KDDTest+ prevalence 0.2972%", "0.2972"),
    ("Largest overall |SMD| 0.5060", "0.5060"),
    ("Cautious wording: consistent with", "consistent with"),
    (
        "Cautious wording: does not prove / not establish causality",
        ("does not prove" in d6.lower())
        or ("does not by itself prove" in d6.lower())
        or ("not establish" in d6.lower())
        or ("do not prove" in d6.lower())
        or ("does **not** establish" in d6.lower()),
    ),
]
shift_rows = []
for label, needle in shift_items:
    ok = needle if isinstance(needle, bool) else needle in d6
    shift_rows.append({"Day 6 finding": label, "Found in Day 6 notebook": bool(ok)})

day9_shift_audit = pd.DataFrame(shift_rows)
display(day9_shift_audit)
print("No additional statistical tests were run.")


## Section 9 — Model comparison integrity (Day 7)

In [ ]:
d7 = val_nbs["07_final_model_comparison.ipynb"]
cmp_items = [
    ("LR class-weighted val Macro F1 0.6985", "0.6985" in d7 or "0.698500" in d7),
    ("LR no SMOTE val Macro F1 0.8560", "0.8560" in d7 or "0.856000" in d7),
    ("LR SMOTE val Macro F1 0.7028", "0.7028" in d7 or "0.702800" in d7),
    ("RF 500 val Macro F1 0.951031", "0.951031" in d7),
    ("RF selected on validation Macro F1", "strongest validation Macro F1" in d7.lower() or "validation-selected" in d7.lower()),
    ("LR no SMOTE KDDTest+ Macro F1 0.5806", "0.5806" in d7),
    ("LR SMOTE KDDTest+ Macro F1 0.5863", "0.5863" in d7),
    ("RF 500 KDDTest+ Macro F1 0.5061", "0.5061" in d7),
    ("Explicit: KDDTest+ cannot change the lock", "cannot retroactively change" in d7.lower() or "does not change" in d7.lower()),
]
day9_comparison_audit = pd.DataFrame(
    [{"Check": k, "Supported": v} for k, v in cmp_items]
)
display(day9_comparison_audit)
print()
print(
    "KDDTest+ comparison is reporting only. This audit does not rank or select "
    "a new final model. Random Forest 500 remains locked."
)


## Section 10 — Experimental-integrity audit

In [ ]:
day9_integrity_checklist = pd.DataFrame(
    [
        {
            "Item": "Final model selected before KDDTest+",
            "Status": "PASS",
            "Evidence/Comment": "Day 5 lock cell and outputs: model locked, then KDDTest+ scored; 'KDDTest+ has not been used for model selection.'",
        },
        {
            "Item": "KDDTest+ evaluation-only",
            "Status": "PASS",
            "Evidence/Comment": "Days 5–8 repeatedly state KDDTest+ is evaluation/reporting only.",
        },
        {
            "Item": "No KDDTest+-based model selection",
            "Status": "PASS",
            "Evidence/Comment": "Day 7 states higher SMOTE test Macro F1 cannot retroactively change the lock.",
        },
        {
            "Item": "No post-test tuning",
            "Status": "PASS",
            "Evidence/Comment": "Day 5/8: 'No post-test tuning was performed.'",
        },
        {
            "Item": "No threshold tuning",
            "Status": "PASS",
            "Evidence/Comment": "Locked config documents no threshold tuning; Day 8 integrity list repeats it.",
        },
        {
            "Item": "No Random Forest retraining after lock",
            "Status": "PASS",
            "Evidence/Comment": "Days 6–8 instruct not to retrain; Day 5 evaluation uses existing rf_final_model. This audit did not retrain.",
        },
        {
            "Item": "No model replacement",
            "Status": "PASS",
            "Evidence/Comment": "Days 7–8 keep RF 500 as locked final model despite higher LR-SMOTE test Macro F1.",
        },
        {
            "Item": "No new train/test split",
            "Status": "PASS",
            "Evidence/Comment": "Day 6 recovers Day 5 split (test_size=0.20, stratify, random_state=42) when labels are missing; no new split design.",
        },
        {
            "Item": "No SMOTE applied to locked RF",
            "Status": "PASS",
            "Evidence/Comment": "Day 5 RF sections state no SMOTE; SMOTE was a Day 4 LR experiment only.",
        },
        {
            "Item": "class_weight remained None",
            "Status": "PASS",
            "Evidence/Comment": "Day 5 lock configuration: class_weight=None.",
        },
        {
            "Item": "Day 6 was post-hoc diagnostic analysis",
            "Status": "PASS",
            "Evidence/Comment": "Day 6 titles and conclusions: post-hoc / diagnostic; no model change.",
        },
        {
            "Item": "Day 7 comparison was reporting only",
            "Status": "PASS",
            "Evidence/Comment": "Day 7: reporting/synthesis; copied metrics; no .fit().",
        },
        {
            "Item": "Day 8 was final reporting/evidence",
            "Status": "PASS",
            "Evidence/Comment": "Day 8: final evaluation and project evidence; no training.",
        },
        {
            "Item": "Day 9 performs no modeling",
            "Status": "PASS",
            "Evidence/Comment": "This notebook: file/Git/text audit only; no sklearn fit/predict.",
        },
        {
            "Item": "Serialized y_test_pred_rf in Day 5 .ipynb",
            "Status": "NOT EXPLICITLY VERIFIABLE",
            "Evidence/Comment": "Day 6 recovery concluded the prediction array was not serialized in notebook JSON; metrics/CM were saved as displayed outputs.",
        },
    ]
)
display(day9_integrity_checklist)


## Section 11 — Final project story audit

1. The project established a **validation protocol on KDDTrain+** (stratified 80/20, `random_state=42`).
2. Multiple approaches were compared (class-weighted LR, LR with/without SMOTE, Random Forest tree counts).
3. **Random Forest with 500 trees** achieved the strongest validation Macro F1 (**0.951031**).
4. It was **locked before KDDTest+ evaluation**.
5. KDDTest+ exposed a **substantial generalization gap** (Macro F1 **0.951031 → 0.5061**).
6. **R2L and U2R** were the major weaknesses (recall **0.964824 → 0.047487** and **0.700000 → 0.059701**).
7. Large **class-composition** and **feature-distribution** differences provide evidence **consistent with dataset/domain shift**.
8. These findings **do not establish causality**.
9. The final model remains **Random Forest 500** because the **predefined validation-based selection procedure** must be respected.

## Section 12 — Final readiness checklist

- [x] Day 1–8 notebooks present *(Day 3 is inside `02_baseline_model.ipynb`; no `03_*.ipynb`)*
- [x] Day 9 audit created
- [x] Locked model clearly documented
- [x] Validation selection clearly documented
- [x] KDDTest+ clearly separated from model selection
- [x] Final metrics consistently reported
- [x] Confusion matrix consistently reported
- [x] Generalization gap documented
- [x] Distribution shift documented
- [x] Limitations documented
- [x] No unsupported causal claims *(wording remains cautious)*
- [x] No accidental retraining/tuning *(this audit; Days 6–8 instructions)*
- [x] Git state audited *(Days 5–8 committed on `main`; Day 9 untracked until the user commits)*

In [ ]:
print("Day 9 reproducibility and project integrity audit complete.")
print("No model was trained, retrained, tuned, modified, or replaced.")
print("KDDTest+ was not used for model selection.")
print("The Random Forest 500-tree model remains the locked final model.")
